# 🚀 Notebook 3A — Maximum DNABERT Fine-Tuning on Perlmutter

## One expensive model. Use the GPUs to train it hard.

Notebook 3 asked:

> **Does adding GPUs make the same workload faster?**

Notebook 3A changes the goal.

We now want to build **one serious DNABERT classifier** and use the GPU allocation as effectively as we reasonably can.

### Main idea

```text
full DNABERT encoder
+ classification head
+ every useful parameter trainable
+ all requested GPUs
+ DDP
+ BF16 mixed precision
+ large per-GPU batch
+ fused optimizer when available
        ↓
maximum useful fine-tuning workload
```

This is **not** a fair 1-GPU vs 2-GPU scaling experiment.

Notebook 3 already taught that.

Notebook 3A is a **throughput-first capstone**.

> **New to Python or machine learning?** Work through
> **`Notebook_Start_Here.ipynb`** first — about an hour, and it teaches
> exactly the Python this notebook uses, plus a glossary you can keep open
> in another tab.

## Before you start

**What this notebook is for:** taking the Notebook 1 job to many GPUs. You measure how much faster it actually gets — not how much faster it should get.

**What you will leave with:**

1. **You have submitted a job to a supercomputer** and watched it run.
2. You can read a Slurm script and say what hardware it asks for.
3. You have measured real speedup against ideal speedup, and can explain
   the gap.
4. You know that more GPUs stops paying off, and can point to where.

**New words you will meet here:** Slurm, sbatch, node, DDP, reservation, strong scaling, parallel efficiency

**If you get lost:** the key idea is one sentence: *the same work, split across more machines, which then have to keep talking to each other.* Everything else is detail.

**Time:** about 2 hours. Jobs take minutes to queue and run. Use the waiting time to read the training script — it is right there in the notebook and readable.

### How to read this notebook

| Marker | Meaning |
|---|---|
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it. |
| 👀 **READ** | Important code. Read the comments and follow the main idea. |
| 🧠 **BUILD IT** | A core concept turned into code. Read this one closely. |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves. |
| 🔲 **YOUR TURN** | A line is left blank on purpose. Write it, then run the check cell below it. |
| ✅ **CHECKPOINT** | Stop and answer before moving on. |

Every notebook in this bootcamp uses these same six markers.

## 🧭 Python survival guide — read this once, then come back when needed

You do **not** need to memorize Python syntax. When you see unfamiliar code, first identify the job it is doing.

| Python word | Plain-English meaning | Tiny example |
|---|---|---|
| **variable** | A name that stores a value | `k = 6` |
| **function** | A reusable mini-program that performs one job | `gc_content(sequence)` |
| **argument** | A value you give to a function | `gc_content("ACGT")` |
| **return** | The value a function gives back | `return gc_fraction` |
| **list** | An ordered collection | `["A", "C", "G", "T"]` |
| **dictionary (`dict`)** | Named values stored as key → value pairs | `{"A": 1, "C": 2}` |
| **DataFrame** | A pandas table: rows are examples, columns are properties | `df.head()` |
| **boolean mask** | A True/False filter that selects rows | `df[df["label"] == 1]` |
| **class** | A blueprint for an object that stores data and behavior together | `class DNASet(...)` |
| **method** | A function that belongs to an object/class | `model.forward(...)` |

### How to read a function

```python
def gc_content(sequence):      # function name + input
    gc = ...                   # work done inside the function
    return gc                  # value sent back
```

Read that as:

> “Given a `sequence`, calculate something called `gc`, then give `gc` back.”

### How to read a class

```python
class ExampleModel(nn.Module):
    def __init__(self):
        ...

    def forward(self, x):
        ...
```

- `__init__` = **what pieces does this object contain?**
- `forward` = **what happens to the input when it moves through the model?**
- `self` = **this particular object**. You normally do not pass it yourself.

Whenever a cell is marked **🔒 RUN ONLY**, focus on the explanation above it rather than every Python detail.



## 🧭 HPC survival guide — the words you will keep seeing

| Term | Plain-English meaning |
|---|---|
| **Slurm** | The scheduler that decides when and where your job runs on Perlmutter. |
| **job** | One request sent to Slurm: hardware + time + commands to execute. |
| **node** | One compute machine. A Perlmutter GPU node contains multiple GPUs. |
| **GPU** | The accelerator doing most of the neural-network math. |
| **QOS** | A Slurm queue/policy that controls what resources a job may request. |
| **reservation** | Compute nodes held for a specific event/time window, such as the bootcamp. |
| **DDP** | *DistributedDataParallel*: multiple Python processes cooperate to train one model. |
| **rank** | The ID of one DDP process: `0`, `1`, `2`, ... |
| **local rank** | Which GPU a process uses on its current node. |
| **world size** | Total number of DDP processes/GPUs participating in the run. |
| **all-reduce** | A communication step that combines gradients from all ranks so every model copy stays synchronized. |
| **BF16** | A 16-bit number format that uses less memory and can run efficiently on A100 GPUs. |
| **throughput** | How much work the system completes per second, e.g. examples/second. |

### Mental model for DDP

```text
rank 0 → GPU 0 → different training examples ┐
rank 1 → GPU 1 → different training examples ├─ all-reduce gradients → synchronized model
rank 2 → GPU 2 → different training examples ┤
rank 3 → GPU 3 → different training examples ┘
```

The important idea is **one model training run**, not four unrelated models.


## Learning objectives

By the end of this notebook, you should be able to:

1. Explain what **full fine-tuning** means.
2. Verify that essentially **100% of the useful model parameters** receive gradients.
3. Use multiple A100 GPUs on one DNABERT training job with DDP.
4. Use BF16 automatic mixed precision to increase GPU throughput.
5. Choose a **local batch size** that makes better use of GPU memory.
6. Measure:
   - trainable parameters,
   - GPU memory usage,
   - examples per second,
   - training time,
   - validation AUROC/AUPRC.
7. Save a trained DNABERT checkpoint for later use.

# 1. What Does “Maximum Fine-Tuning” Mean?

There are two very different ideas:

### Parameter-efficient fine-tuning

```text
freeze most of DNABERT
↓
train only a small subset
```

Examples include training only the classifier or only the last few layers.

### Full fine-tuning

```text
embeddings
+
all Transformer encoder layers
+
classification head
        ↓
all optimized together
```

Notebook 3A uses **full fine-tuning**.

The BERT pooler is not included because our classifier does not use `pooler_output`.
Instead, we directly classify the final hidden state of the `[CLS]` token.

That means the model contains **only parameters that participate in the task**.

## Why not increase `max_length` to 512?

Our DNA sequences are about **200 bp**.

With overlapping 6-mers:

```text
200 bp
↓
195 overlapping 6-mers
↓
special tokens
↓
fits inside max_length = 256
```

Using 512 would mostly add padding.

That would consume more memory without adding biological information.

So we spend our GPU budget on:

```text
more trainable parameters
larger batches
more epochs
multiple GPUs
mixed precision
```

—not unnecessary padding.

# 2. Setup

In [ ]:
import os
import sys
import json
import time
import shlex
import subprocess
import py_compile

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# All bootcamp notebooks resolve the SAME folder, so a dataset built in
# Notebook 0 is found by Notebooks 1-3 even if you launch them from
# elsewhere. Override by setting the DNA_BOOTCAMP_HOME environment variable.
PROJECT_DIR = Path(os.environ.get("DNA_BOOTCAMP_HOME", ".")).expanduser().resolve()
print("📁 Project directory:", PROJECT_DIR)

DATA_DIR = Path(
    "/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example"
)
RESULTS_DIR = (PROJECT_DIR / "notebook3a_results")

SCRIPTS_DIR = (PROJECT_DIR / "notebook3a_scripts")

SLURM_LOG_DIR = (RESULTS_DIR / "slurm_logs")

for directory in [RESULTS_DIR, SCRIPTS_DIR, SLURM_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# IMPORTANT:
# Do not assume the current Jupyter kernel is the same
# Python environment that should run the Slurm training job.
#
# Test likely dna-llm interpreters and choose the first one
# that can import the packages required by Notebook 3A.
# ---------------------------------------------------------

PYTHON_CANDIDATES = [
    # Shared project environment — best for a bootcamp if available.
    Path("/global/cfs/cdirs/m4388/envs/dna-llm/bin/python"),

    # User-local environment used in earlier Notebook 3 runs.
    Path.home() / ".conda" / "envs" / "dna-llm" / "bin" / "python",

    # Current Jupyter kernel as a final candidate.
    Path(sys.executable)]


def python_environment_check(python_path,):
    """
    Return (works, message).

    A usable training interpreter must import all packages
    needed by the DNABERT Slurm program.
    """

    if not python_path.exists():
        return (False, "path does not exist")

    command = [str(python_path), "-c", ("import sys; " "import torch; "
            "import transformers; " "import sklearn; " "import pandas; "
            "import numpy; " "print(sys.executable); "
            "print('torch=' + torch.__version__); "
            "print('transformers=' + transformers.__version__)")]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        message = (result.stderr.strip() or result.stdout.strip()
            or "import check failed")

        return (False, message)

    return (True, result.stdout.strip())


NOTEBOOK_PYTHON = None

seen = set()

for candidate in PYTHON_CANDIDATES:
    candidate = candidate.resolve()

    if candidate in seen:
        continue

    seen.add(candidate)

    works, message = (python_environment_check(candidate))

    print()
    print("Python candidate:", candidate)

    if works:
        print("✅ Required packages available")

        print(message)

        if NOTEBOOK_PYTHON is None:
            NOTEBOOK_PYTHON = str(candidate)

    else:
        print("❌ Not usable for DNABERT")

        print(message.splitlines()[-1] if message else "unknown error")


if NOTEBOOK_PYTHON is None:
    raise RuntimeError("Notebook 3A could not find a Python interpreter "
        "that imports torch, transformers, sklearn, pandas, "
        "and numpy. Select/repair the dna-llm environment "
        "before submitting the GPU job.")


print()
print("✅ Slurm training Python:", NOTEBOOK_PYTHON)

print("Data directory:", DATA_DIR)

print("Results directory:", RESULTS_DIR)

# 3. Student GPU Configuration

During the bootcamp, the default below requests **one full Perlmutter GPU node = 4 A100 GPUs** using the day's bootcamp reservation.

For testing outside bootcamp hours, students can switch:

```python
RUN_MODE = "shared"
GPU_COUNT = 1   # or 2
```

## Throughput-first batch design

Notebook 3A specifies a **local batch per GPU**.

```text
4 GPUs × local batch 32 = global batch 128
```

Each rank handles its local batch, and DDP synchronizes gradients so the four GPU replicas behave like one training run.

This notebook is about using the allocated hardware for one substantial full-fine-tuning run, not minimizing resource use.


In [ ]:
# ✏️ EDIT ME — where you are running, and on how much hardware

NERSC_ACCOUNT = "m4388"

# "shared"   -> testing outside bootcamp hours (1-2 GPUs, no reservation)
# "bootcamp" -> during the bootcamp (uses that day's reservation)
RUN_MODE = "bootcamp"

# Only used when RUN_MODE = "bootcamp". Which day of the bootcamp is it?
BOOTCAMP_DAY = 1

# How many GPUs for the maximum run.
#   shared   : 1 or 2
#   bootcamp : any multiple of 4 (whole nodes), or 1-4 on a single node
GPU_COUNT = 4

# Examples per GPU per step. Raise this if GPU memory stays low.
LOCAL_BATCH_SIZE = 32
GLOBAL_BATCH_SIZE = GPU_COUNT * LOCAL_BATCH_SIZE

MAX_CONFIG = {
    "run_name": "dnabert_maximum_full_finetune",
    "gpus": GPU_COUNT,

    # More training than the short demonstration runs in Notebook 1.
    "epochs": 10,

    # Derived so every GPU gets LOCAL_BATCH_SIZE examples.
    "batch_size": GLOBAL_BATCH_SIZE,

    # Full BERT fine-tuning needs a smaller LR than a randomly
    # initialized classifier head.
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "dropout": 0.10,

    # 200 bp -> 195 six-mers + [CLS] + [SEP] = 197 tokens. Padding to
    # 256 would waste 23% of every attention computation on nothing.
    "max_length": 200,

    # A100-oriented precision mode.
    "precision": "bf16",
    "seed": 42,
}

pd.Series(MAX_CONFIG, name="value")

## Where are you running this?

A Slurm job needs to know two different things, and students mix them up
constantly:

1. **How many GPUs do you want?** — that is `GPU_COUNT`.
2. **Which pool of machines are you allowed to take them from?** — that is
   the QOS, and during the bootcamp, a *reservation*.

A Perlmutter GPU node has **4 A100 GPUs**. So asking for 8 GPUs is really
asking for 2 whole nodes. The cell below does that arithmetic for you.

| `RUN_MODE` | When | QOS | GPUs you can ask for |
|---|---|---|---|
| `"shared"` | Testing, outside bootcamp hours | `shared` | 1 or 2, on one shared node |
| `"bootcamp"` | During the bootcamp | `regular` + that day's reservation | whole nodes, up to the day's limit |

**The bootcamp reservations:**

| Day | Reservation | Window | Nodes | GPUs |
|---|---|---|---:|---:|
| 1 | `bootcamp_day1` | 11:00 – 22:00 | 20 | 80 |
| 2 | `bootcamp_day2` | 11:00 – 22:00 | 30 | 120 |
| 3 | `bootcamp_day3` | 11:00 – 22:00 | 30 | 120 |
| 4 | `bootcamp_day4` | 08:00 – 00:00 | 40 | 160 |
| 5 | `bootcamp_day5` | 08:00 – 11:00 | 30 | 120 |

A reservation is **not** extra hardware you are entitled to on top of the
queue — it is a block of nodes held aside so your job does not wait behind
everyone else's. Outside the window, the reservation does not exist and the
job will be rejected.

In [ ]:
# 🔒 RUN ONLY — turn "I want N GPUs" into Slurm flags that actually get you N

import math

GPUS_PER_NODE = 4          # a Perlmutter GPU node has 4 A100s

# name, nodes held, window
BOOTCAMP_RESERVATIONS = {
    1: ("bootcamp_day1", 20, "11:00-22:00"),
    2: ("bootcamp_day2", 30, "11:00-22:00"),
    3: ("bootcamp_day3", 30, "11:00-22:00"),
    4: ("bootcamp_day4", 40, "08:00-00:00"),
    5: ("bootcamp_day5", 30, "08:00-11:00"),
}


def resolve_slurm(gpu_count, run_mode=None, day=None):
    """Return the Slurm settings needed to obtain `gpu_count` GPUs."""
    run_mode = RUN_MODE if run_mode is None else run_mode
    day = BOOTCAMP_DAY if day is None else day
    gpu_count = int(gpu_count)

    if gpu_count < 1:
        raise ValueError("Request at least one GPU.")

    if run_mode == "shared":
        if gpu_count > 2:
            raise ValueError(
                f"RUN_MODE='shared' is for testing and allows 1-2 GPUs, "
                f"but GPU_COUNT={gpu_count}. During the bootcamp set "
                f"RUN_MODE='bootcamp' and BOOTCAMP_DAY."
            )
        return {"qos": "shared", "reservation": None, "nodes": 1,
                "tasks_per_node": gpu_count, "gpus_per_node": gpu_count,
                "gpus": gpu_count, "window": "any"}

    if run_mode == "bootcamp":
        if day not in BOOTCAMP_RESERVATIONS:
            raise ValueError(f"BOOTCAMP_DAY must be one of "
                             f"{sorted(BOOTCAMP_RESERVATIONS)}, got {day}.")
        name, max_nodes, window = BOOTCAMP_RESERVATIONS[day]
        nodes = math.ceil(gpu_count / GPUS_PER_NODE)

        if nodes > 1 and gpu_count % GPUS_PER_NODE != 0:
            raise ValueError(
                f"A multi-node run must use whole nodes: GPU_COUNT must be a "
                f"multiple of {GPUS_PER_NODE}, got {gpu_count}. Try "
                f"{nodes * GPUS_PER_NODE}."
            )
        if nodes > max_nodes:
            raise ValueError(
                f"{gpu_count} GPUs needs {nodes} nodes, but {name} only holds "
                f"{max_nodes} ({max_nodes * GPUS_PER_NODE} GPUs)."
            )

        per_node = gpu_count if nodes == 1 else GPUS_PER_NODE
        return {"qos": "regular", "reservation": name, "nodes": nodes,
                "tasks_per_node": per_node, "gpus_per_node": per_node,
                "gpus": gpu_count, "window": window}

    raise ValueError("RUN_MODE must be 'shared' or 'bootcamp'.")


PLAN = resolve_slurm(GPU_COUNT)

print("Slurm plan for", GPU_COUNT, "GPU(s)")
print("-" * 42)
for key in ["qos", "reservation", "nodes", "tasks_per_node",
            "gpus_per_node", "window"]:
    print(f"  {key:<15}: {PLAN[key]}")
print()
print(f"  {PLAN['nodes']} node(s) x {PLAN['gpus_per_node']} GPU(s) "
      f"= {PLAN['gpus']} GPU(s) total")
if PLAN["reservation"]:
    print(f"  ⚠️  {PLAN['reservation']} only exists during {PLAN['window']}.")

# 4. How Notebook 3A Uses the GPU

## DDP

Every GPU receives a complete copy of DNABERT.

```text
GPU 0 / rank 0 ─┐
                 ├─ synchronize gradients ─→ one model
GPU 1 / rank 1 ─┘
```

`DistributedSampler` gives different training examples to each rank.

After backward propagation, DDP uses an **all-reduce** so the replicas receive synchronized gradients.

---

## BF16 mixed precision

Model weights remain in normal training precision, while compatible forward operations are automatically executed in BF16.

Conceptually:

```text
FP32 model parameters
        ↓
autocast
        ↓
many matrix operations use BF16
        ↓
less memory + faster tensor-core-friendly computation
        ↓
backward + optimizer
```

We do **not** manually convert the whole model to BF16.

---

## DDP memory optimization

The DDP wrapper uses:

```python
gradient_as_bucket_view=True
```

so gradient tensors can share storage with DDP communication buckets after the first iteration.

The goal is to reduce unnecessary gradient-memory duplication.

# 5. Parameter Goal

Notebook 3A removes BERT's unused pooling layer at model construction:

```python
BertModel.from_pretrained(
    MODEL_NAME,
    add_pooling_layer=False,
)
```

Then every remaining parameter is trainable.

The audit we want is:

```text
useful trainable parameters
---------------------------
total model parameters

≈ 100%
```

If that value is below 100%, students should investigate why.

# 6. Write the DDP Helper Module

**📄 `notebook3a_scripts/ddp_common.py`** — this is the real program Slurm will run on the GPU nodes.

It is written to disk by the `%%writefile` magic below, so what you read here
is exactly what executes. Edit the cell and re-run it to change the job.

## 🧩 Function map — `ddp_common.py`

This small file contains reusable distributed-training helpers. You only need the purpose of each one:

| Function | Plain-English job |
|---|---|
| `setup_distributed` | Reads Slurm/DDP environment variables, assigns this process to a GPU, and connects all ranks into one NCCL process group. |
| `cleanup_distributed` | Disconnects the process group cleanly when training ends. |
| `reduce_training_stats` | Adds loss/correct/example counts across ranks so the printed training metric represents **all GPUs**, not only rank 0. |
| `binary_metrics` | Computes accuracy, precision, recall, specificity, F1, AUROC, AUPRC, and confusion counts. |

**Process group:** the set of DDP processes that are allowed to communicate with one another.

The most important line conceptually is:

```text
each rank has local statistics → all_reduce → one global statistic
```


In [ ]:
%%writefile notebook3a_scripts/ddp_common.py
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist

from sklearn.metrics import (roc_auc_score, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score)


def setup_distributed():
    rank = int(os.environ.get("RANK", "0"))

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))

    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    distributed = (world_size > 1)

    visible_gpu_count = (torch.cuda.device_count())

    cuda_visible_devices = (os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"
        ))

    print(f"[DDP setup] " f"RANK={rank} | " f"LOCAL_RANK={local_rank} | "
        f"WORLD_SIZE={world_size} | " f"visible_GPUs={visible_gpu_count} | "
        f"CUDA_VISIBLE_DEVICES=" f"{cuda_visible_devices}", flush=True)

    if visible_gpu_count < world_size:
        raise RuntimeError(f"Rank {rank} sees only "
            f"{visible_gpu_count} GPU(s), " f"but WORLD_SIZE={world_size}.")

    if local_rank >= visible_gpu_count:
        raise RuntimeError(f"LOCAL_RANK={local_rank}, but only "
            f"{visible_gpu_count} GPU ordinal(s) " "are visible.")

    torch.cuda.set_device(local_rank)

    device = torch.device("cuda", local_rank)

    print(f"[GPU mapping] " f"rank={rank} → cuda:{local_rank} | "
        f"{torch.cuda.get_device_name(local_rank)}", flush=True)

    if distributed:
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            rank=rank,
            world_size=world_size,
            device_id=device,
        )

    return (distributed, rank, world_size, local_rank, device)


def cleanup_distributed(distributed):
    if (distributed and dist.is_initialized()):
        dist.destroy_process_group()


def reduce_training_stats(loss_sum, correct, n, device, distributed):
    values = torch.tensor([loss_sum, correct, n], dtype=torch.float64,
        device=device)

    if distributed:
        dist.all_reduce(values, op=dist.ReduceOp.SUM)

    loss_total, correct_total, n_total = (values.tolist())

    return (loss_total / n_total, correct_total / n_total, int(n_total))


def binary_metrics(y_true, y_pred, scores):
    tn, fp, fn, tp = (confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel())

    return {"accuracy": ((tp + tn) / (tp + tn + fp + fn)),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": (tn / (tn + fp) if (tn + fp) else np.nan),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, scores),
        "auprc": average_precision_score(y_true, scores),
        "true_negative": int(tn), "false_positive": int(fp),
        "false_negative": int(fn), "true_positive": int(tp)}

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
DDP_COMMON = SCRIPTS_DIR / "ddp_common.py"

if not DDP_COMMON.exists():
    raise FileNotFoundError(
        f"Expected {DDP_COMMON} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3a_scripts/ddp_common.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(DDP_COMMON), doraise=True)

print("✅", DDP_COMMON)
print(f"   {len(DDP_COMMON.read_text().splitlines()):,} lines, valid Python")


# 7. Build the Maximum DNABERT Training Program

**📄 `notebook3a_scripts/maximum_dnabert.py`** — this is the real program Slurm will run on the GPU nodes.

It is written to disk by the `%%writefile` magic below, so what you read here
is exactly what executes. Edit the cell and re-run it to change the job.

## 🧩 Map of the training program — do not read 600 lines as one block

The next cell writes a complete Python training program. Read it in these chunks:

| Part | Main names | What it does |
|---|---|---|
| **Data** | `clean_split`, `kmer_sentence`, `DNASet` | Recreates the same clean train/validation data used earlier and converts DNA to DNABERT 6-mer input. |
| **Model** | `MaximumDNABertClassifier` | Loads pretrained DNABERT **without the unused pooler** and adds the Binding/Background classifier. |
| **Audit** | `parameter_group_table` | Counts which parts of the model contain trainable parameters. |
| **Optimization** | `build_optimizer` | Creates AdamW and separates parameters that should/shouldn't receive weight decay. |
| **Training** | `train_epoch` | Forward → loss → backward → gradient clipping → optimizer/scheduler step. |
| **Validation** | `evaluate` | Makes predictions without updating parameters. |
| **Coordinator** | `main` | Parses settings, creates all objects, runs epochs, gathers DDP results, saves metrics/checkpoint. |

### New words

- **AdamW:** the optimizer used to update model parameters.
- **weight decay:** a regularization term that discourages unnecessarily large weights.
- **warmup:** start with a smaller learning rate and increase it gradually at the beginning of training.
- **gradient clipping:** limits unusually large gradients to make training more stable.
- **checkpoint:** a saved copy of trained model parameters that can be loaded later.
- **autocast:** PyTorch automatically uses BF16 for operations where it is appropriate.

Students should understand the **flow**, not memorize the infrastructure.


In [ ]:
%%writefile notebook3a_scripts/maximum_dnabert.py
import argparse
import json
import math
import time
from pathlib import Path

# Keep student-facing logs focused on training results.
# Library errors remain visible; routine advisory messages are suppressed.
import warnings

from huggingface_hub import logging as hf_hub_logging
from transformers.utils import logging as transformers_logging

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"torch\.distributed\.c10d_logger",
)

hf_hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.parallel import DistributedDataParallel as DDP

from torch.utils.data import Dataset, DataLoader

from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import train_test_split

from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import (AutoTokenizer, BertModel,
    get_linear_schedule_with_warmup)

from ddp_common import (setup_distributed, cleanup_distributed,
    reduce_training_stats, binary_metrics)


MODEL_NAME = ("zhihan1996/DNA_bert_6")


def clean_split(data_dir, seed):
    data_dir = Path(data_dir)

    seqs = [line.strip().upper() for line in open(data_dir / "seqs.txt")
        if line.strip()]

    labels = [int(line.strip()) for line in open(data_dir / "labels.txt")
        if line.strip()]

    df = pd.DataFrame({"sequence": seqs, "label": labels})

    df["length"] = (df["sequence"].str.len())

    expected_length = int(df["length"].mode().iloc[0])

    valid = (df["length"].eq(expected_length) & df["sequence"].apply(
            lambda seq: set(seq) <= set("ACGT")))

    conflicts = set(df.groupby("sequence")["label"] .nunique() .loc[
            lambda values: values > 1] .index)

    clean_df = (df[valid & ~df["sequence"].isin(conflicts)] .drop_duplicates(
            "sequence") .reset_index(drop=True))

    train_df, val_df = (train_test_split(clean_df, test_size=0.20,
            random_state=seed, stratify=(clean_df["label"])))

    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True))


def kmer_sentence(seq, k=6):
    return " ".join(seq[i:i+k] for i in range(len(seq) - k + 1))


class DNASet(Dataset):
    def __init__(self, df, tokenizer, max_length):
        sentences = [kmer_sentence(seq) for seq in df["sequence"]]

        encoded = tokenizer(sentences, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt")

        self.inputs = (encoded["input_ids"])

        self.masks = (encoded["attention_mask"])

        self.labels = torch.tensor(df["label"].to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input": self.inputs[idx], "attention_mask": self.masks[idx],
 "label": self.labels[idx]}


class MaximumDNABertClassifier(nn.Module):
    def __init__(self, dropout):
        super().__init__()

        # IMPORTANT:
        # We remove the BERT pooler entirely because
        # this classifier directly uses the final
        # hidden state of the [CLS] token.
        self.bert = (BertModel.from_pretrained(MODEL_NAME,
                add_pooling_layer=False))

        hidden_size = (self.bert.config.hidden_size)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(hidden_size, 2)

        # FULL FINE-TUNING:
        # every remaining parameter receives gradients.
        for parameter in (self.parameters()):
            parameter.requires_grad = True

    def forward(self, x, attention_mask):
        output = self.bert(input_ids=x, attention_mask=(attention_mask))

        cls_vector = (output .last_hidden_state[:, 0, :])

        return self.classifier(self.dropout(cls_vector))


def parameter_group_table(model,):
    rows = []

    for name, parameter in (model.named_parameters()):
        if name.startswith("bert.embeddings"):
            group = ("BERT embeddings")

        elif ("bert.encoder.layer." in name):
            layer_number = (name.split("bert.encoder.layer.")[1] .split(".")[0]
            )

            group = ("BERT layer " + layer_number)

        elif name.startswith("classifier"):
            group = ("classification head")

        else:
            group = "other"

        rows.append({"parameter": name, "group": group, "numel": (
                parameter.numel()), "trainable": bool(parameter.requires_grad)
        })

    detail = pd.DataFrame(rows)

    grouped = (detail.groupby(["group", "trainable"], as_index=False
        )["numel"] .sum())

    return (detail, grouped)


def build_optimizer(model, learning_rate, weight_decay):
    no_decay_terms = ("bias", "LayerNorm.weight")

    decay_parameters = []
    no_decay_parameters = []

    for name, parameter in (model.named_parameters()):
        if any(term in name for term in no_decay_terms):
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    groups = [{"params": decay_parameters,
 "weight_decay": weight_decay}, {"params": no_decay_parameters,
 "weight_decay": 0.0}]

    try:
        optimizer = (torch.optim.AdamW(groups, lr=learning_rate, fused=True))

        optimizer_mode = ("AdamW fused=True")

    except (TypeError, RuntimeError):
        optimizer = (torch.optim.AdamW(groups, lr=learning_rate))

        optimizer_mode = ("AdamW standard")

    return (optimizer, optimizer_mode)


def train_epoch(model, optimizer, scheduler, loader, sampler, epoch, device,
    distributed):
    model.train()

    if sampler is not None:
        sampler.set_epoch(epoch)

    loss_sum = 0.0
    correct = 0
    n = 0

    for batch_index, batch in enumerate(loader):
        x = batch["input"].to(device, non_blocking=True)

        mask = batch["attention_mask"].to(device, non_blocking=True)

        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)

            loss = (F.cross_entropy(logits, labels))

        loss.backward()

        # Full-fine-tuning audit:
        # on the first batch every trainable parameter
        # should have a gradient.
        if (epoch == 1 and batch_index == 0):
            missing_grads = [name for name, parameter
                in model.named_parameters() if (parameter.requires_grad
                    and parameter.grad is None)]

            if missing_grads:
                raise RuntimeError("Trainable parameters "
                    "without gradients: " + ", ".join(missing_grads))

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        predictions = (logits.argmax(dim=1))

        loss_sum += (loss.item() * len(labels))

        correct += (predictions == labels).sum().item()

        n += len(labels)

    return reduce_training_stats(loss_sum, correct, n, device, distributed)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    loss_sum = 0.0
    correct = 0
    n = 0

    scores = []
    predictions = []
    truth = []

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)

        mask = batch["attention_mask"].to(device, non_blocking=True)

        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)

            loss = (F.cross_entropy(logits, labels))

        pred = logits.argmax(dim=1)

        prob = torch.softmax(logits.float(), dim=1)[:, 1]

        loss_sum += (loss.item() * len(labels))

        correct += (pred == labels).sum().item()

        n += len(labels)

        scores.extend(prob.cpu().numpy())

        predictions.extend(pred.cpu().numpy())

        truth.extend(labels.cpu().numpy())

    return {"loss": loss_sum / n, "accuracy": correct / n,
 "scores": np.asarray(scores), "predictions": np.asarray(predictions),
 "true": np.asarray(truth)}


def main():
    parser = (argparse.ArgumentParser())

    parser.add_argument("--data_dir", required=True)

    parser.add_argument("--output_dir", required=True)

    parser.add_argument("--run_name", required=True)

    parser.add_argument("--epochs", type=int, required=True)

    parser.add_argument("--batch_size", type=int, required=True)

    parser.add_argument("--learning_rate", type=float, required=True)

    parser.add_argument("--weight_decay", type=float, required=True)

    parser.add_argument("--warmup_ratio", type=float, required=True)

    parser.add_argument("--dropout", type=float, required=True)

    # 200 bp -> 195 six-mers + [CLS] + [SEP] = 197 tokens.
    parser.add_argument("--max_length", type=int, default=200)

    parser.add_argument("--precision", choices=["bf16",], default="bf16")

    parser.add_argument("--seed", type=int, default=42)

    args = (parser.parse_args())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required.")

    if not (torch.cuda.is_bf16_supported()):
        raise RuntimeError("Notebook 3A expects " "BF16-capable GPUs.")

    (distributed, rank, world_size, local_rank, device) = setup_distributed()

    try:
        if (args.batch_size % world_size != 0):
            raise ValueError("Global batch size " "must be divisible "
                "by GPU count.")

        local_batch_size = (args.batch_size // world_size)

        np.random.seed(args.seed)

        torch.manual_seed(args.seed)

        torch.cuda.manual_seed_all(args.seed)

        # Allow optimized matmul paths for remaining FP32 ops.
        torch.set_float32_matmul_precision("high")

        train_df, val_df = (clean_split(args.data_dir, args.seed))

        tokenizer = (AutoTokenizer .from_pretrained(MODEL_NAME))

        train_dataset = DNASet(train_df, tokenizer, args.max_length)

        val_dataset = DNASet(val_df, tokenizer, args.max_length)

        train_sampler = None

        if distributed:
            train_sampler = (DistributedSampler(train_dataset, num_replicas=(
                        world_size), rank=rank, shuffle=True, seed=args.seed))

        loader_workers = min(4, max(1, (int(__import__("os").environ.get(
                            "SLURM_CPUS_PER_TASK", "4")) // 4)))

        train_loader = (DataLoader(train_dataset, batch_size=(local_batch_size
                ), shuffle=(train_sampler is None), sampler=(train_sampler),
                pin_memory=True, num_workers=(loader_workers),
                persistent_workers=(loader_workers > 0)))

        # Rank 0 performs validation.
        val_loader = (DataLoader(val_dataset, batch_size=(local_batch_size),
                shuffle=False, pin_memory=True, num_workers=(loader_workers),
                persistent_workers=(loader_workers > 0)))

        model = (MaximumDNABertClassifier(dropout=(args.dropout)) .to(device))

        total_parameters = sum(p.numel() for p in model.parameters())

        trainable_parameters = sum(p.numel() for p in model.parameters()
            if p.requires_grad)

        trainable_percent = (100.0 * trainable_parameters / total_parameters)

        if (trainable_parameters != total_parameters):
            raise RuntimeError("Maximum fine-tuning " "requires all remaining "
                "model parameters to " "be trainable.")

        parameter_detail, (parameter_groups) = parameter_group_table(model)

        if distributed:
            model = DDP(model, device_ids=[device.index], output_device=(
                    device.index), gradient_as_bucket_view=True,
                static_graph=True)

        (optimizer, optimizer_mode) = build_optimizer(model,
            args.learning_rate, args.weight_decay)

        total_steps = (len(train_loader) * args.epochs)

        warmup_steps = int(total_steps * args.warmup_ratio)

        scheduler = (get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=(warmup_steps), num_training_steps=(
                    total_steps)))

        # Different dropout streams after model sync.
        torch.manual_seed(args.seed + rank)

        torch.cuda.manual_seed_all(args.seed + rank)

        if rank == 0:
            print("\n=== MAXIMUM DNABERT ===", flush=True)

            print(f"GPUs: {world_size}", flush=True)

            print(f"Precision: " f"{args.precision}", flush=True)

            print(f"Global batch: " f"{args.batch_size}", flush=True)

            print(f"Local batch/GPU: " f"{local_batch_size}", flush=True)

            print(f"Trainable parameters: " f"{trainable_parameters:,}",
                flush=True)

            print(f"Total parameters: " f"{total_parameters:,}", flush=True)

            print(f"Trainable percent: " f"{trainable_percent:.2f}%",
                flush=True)

            print(f"Optimizer: " f"{optimizer_mode}", flush=True)

            print(f"Steps/epoch/rank: " f"{len(train_loader)}", flush=True)

        if distributed:
            dist.barrier()

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(device)

        torch.cuda.synchronize(device)

        training_start = (time.time())

        history = []
        final_val = None
        processed_examples = 0

        # Keep the BEST epoch, not the last one. A 10-epoch run has plenty of
        # chances to end on a worse epoch than its peak, and reporting the
        # last one understates the model you actually trained.
        best_auroc = -1.0
        best_val = None
        best_epoch = 0
        best_state = None

        for epoch in range(1, args.epochs + 1):
            epoch_start = (time.time())

            (train_loss, train_accuracy, global_examples_seen) = train_epoch(
                model, optimizer, scheduler, train_loader, train_sampler,
                epoch, device, distributed)

            processed_examples += (global_examples_seen)

            if distributed:
                dist.barrier()

            if rank == 0:
                eval_model = (model.module if distributed else model)

                final_val = evaluate(eval_model, val_loader, device)

                val_auroc = (roc_auc_score(final_val["true"], final_val[
                            "scores"]))

                val_auprc = (average_precision_score(final_val["true"],
                        final_val["scores"]))

                epoch_seconds = (time.time() - epoch_start)

                row = {"epoch": epoch, "train_loss": train_loss,
 "train_accuracy": train_accuracy, "val_loss": final_val["loss"],
 "val_accuracy": final_val["accuracy"], "val_auroc": val_auroc,
 "val_auprc": val_auprc, "epoch_seconds": epoch_seconds,
 "learning_rate": optimizer.param_groups[0]["lr"]}

                history.append(row)

                if val_auroc > best_auroc:
                    best_auroc = val_auroc
                    best_val = final_val
                    best_epoch = epoch
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in eval_model.state_dict().items()}

                if final_val["loss"] > 0.69 and val_auroc < 0.55:
                    print("  \U0001F6A8 Collapsed to chance "
                          "(val loss near ln(2)). Lower --learning_rate.",
                          flush=True)

                print(f"Epoch " f"{epoch}/" f"{args.epochs} | " f"train loss="
                    f"{train_loss:.4f} | " f"train acc="
                    f"{train_accuracy:.3f} | " f"val loss="
                    f"{final_val['loss']:.4f} | " f"AUROC="
                    f"{val_auroc:.4f} | " f"AUPRC=" f"{val_auprc:.4f} | "
                    f"{epoch_seconds:.1f}s", flush=True)

            if distributed:
                dist.barrier()

        torch.cuda.synchronize(device)

        training_time = (time.time() - training_start)

        local_peak_bytes = (torch.cuda .max_memory_allocated(device))

        peak_tensor = torch.tensor([float(local_peak_bytes)],
            dtype=torch.float64, device=device)

        if distributed:
            dist.all_reduce(peak_tensor, op=dist.ReduceOp.MAX)

        max_peak_bytes = (peak_tensor.item())

        total_memory_bytes = (torch.cuda .get_device_properties(device)
            .total_memory)

        peak_memory_gb = (max_peak_bytes / (1024 ** 3))

        total_memory_gb = (total_memory_bytes / (1024 ** 3))

        peak_memory_percent = (100.0 * max_peak_bytes / total_memory_bytes)

        if rank == 0:
            history_df = (pd.DataFrame(history))

            best_row = (history_df.loc[history_df["val_auroc"].idxmax()])

            # Restore the best epoch before reporting, checkpointing or
            # writing predictions, so everything below describes the model
            # that was actually kept.
            eval_model = (model.module if distributed else model)

            if best_state is not None:
                eval_model.load_state_dict(best_state)
                final_val = best_val
                print(f"\u21A9\uFE0F  Restored weights from epoch "
                      f"{best_epoch} (AUROC {best_auroc:.4f}).", flush=True)

            metrics = (binary_metrics(final_val["true"], final_val[
                        "predictions"], final_val["scores"]))

            predictions_df = (val_df[["sequence", "label"]] .copy())

            predictions_df["predicted_label"] = (final_val["predictions"])

            predictions_df["binding_probability"] = (final_val["scores"])

            predictions_df["correct"] = (predictions_df["label"]
                == predictions_df["predicted_label"])

            # Use the actual number of examples processed by
            # all DDP ranks. DistributedSampler can pad a small
            # number of samples to make rank lengths equal.
            examples_per_second = (processed_examples / training_time)

            output_dir = Path(args.output_dir)

            output_dir.mkdir(parents=True, exist_ok=True)

            prefix = (output_dir / args.run_name)

            history_df.to_csv(str(prefix) + "_history.csv", index=False)

            predictions_df.to_csv(str(prefix) + "_predictions.csv",
                index=False)

            parameter_detail.to_csv(str(prefix) + "_parameters.csv",
                index=False)

            parameter_groups.to_csv(str(prefix) + "_parameter_groups.csv",
                index=False)

            eval_model = (model.module if distributed else model)

            # Save after the training timer so checkpoint I/O
            # does not distort the reported training time.
            checkpoint_path = (str(prefix) + "_final_checkpoint.pt")

            torch.save({"model_name": MODEL_NAME,
 "state_dict": eval_model.state_dict(), "epoch": int(best_epoch),
 "max_length": args.max_length, "dropout": args.dropout}, checkpoint_path)

            summary = {"family": "dnabert", "run_name": args.run_name,
 "model_name": MODEL_NAME, "strategy": "maximum_full_finetuning",
 "distributed": bool(distributed), "num_gpus": int(world_size),
 "precision": args.precision, "epochs": int(args.epochs),
 "global_batch_size": int(args.batch_size),
 "local_batch_size": int(local_batch_size),
 "learning_rate": float(args.learning_rate),
 "weight_decay": float(args.weight_decay),
 "warmup_ratio": float(args.warmup_ratio), "dropout": float(args.dropout),
 "max_length": int(args.max_length), "random_seed": int(args.seed),
 "optimizer": optimizer_mode,
 "trainable_parameters": int(trainable_parameters),
 "total_parameters": int(total_parameters),
 "trainable_percent": float(trainable_percent),
 "best_val_auroc": float(best_row["val_auroc"]),
 "best_epoch": int(best_row["epoch"]),
 "final_val_accuracy": float(metrics["accuracy"]),
 "final_val_precision": float(metrics["precision"]),
 "final_val_recall": float(metrics["recall"]),
 "final_val_specificity": float(metrics["specificity"]),
 "final_val_f1": float(metrics["f1"]),
 "final_val_auroc": float(metrics["auroc"]),
 "final_val_auprc": float(metrics["auprc"]),
 "training_time_seconds": float(training_time),
 "examples_per_second": float(examples_per_second),
 "peak_gpu_memory_gb": float(peak_memory_gb),
 "gpu_memory_capacity_gb": float(total_memory_gb),
 "peak_gpu_memory_percent": float(peak_memory_percent),
 "checkpoint_path": checkpoint_path}

            with open(str(prefix) + "_summary.json", "w") as handle:
                json.dump(summary, handle, indent=2)

            print()
            print("=== FINAL REPORT ===", flush=True)

            print(f"Best AUROC: " f"{summary['best_val_auroc']:.4f}",
                flush=True)

            print(f"Training time: " f"{training_time:.1f}s", flush=True)

            print(f"Examples/sec: " f"{examples_per_second:.1f}", flush=True)

            print(f"Peak GPU memory: " f"{peak_memory_gb:.2f} / "
                f"{total_memory_gb:.2f} GB " f"({peak_memory_percent:.1f}%)",
                flush=True)

            print(f"Checkpoint: " f"{checkpoint_path}", flush=True)

    finally:
        cleanup_distributed(distributed)


if __name__ == "__main__":
    main()

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
MAX_SCRIPT = SCRIPTS_DIR / "maximum_dnabert.py"

if not MAX_SCRIPT.exists():
    raise FileNotFoundError(
        f"Expected {MAX_SCRIPT} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3a_scripts/maximum_dnabert.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(MAX_SCRIPT), doraise=True)

print("✅", MAX_SCRIPT)
print(f"   {len(MAX_SCRIPT.read_text().splitlines()):,} lines, valid Python")


## Key code to recognize

Students should be able to explain these four decisions:

### 1. Remove the unused pooler

```python
BertModel.from_pretrained(
    MODEL_NAME,
    add_pooling_layer=False,
)
```

### 2. Train everything that remains

```python
for parameter in self.parameters():
    parameter.requires_grad = True
```

### 3. Use BF16 for compatible forward operations

```python
with torch.amp.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    ...
```

### 4. Use DDP memory-efficient gradient buckets

```python
DDP(
    model,
    ...,
    gradient_as_bucket_view=True,
    static_graph=True,
)
```

# 8. SLURM Submission Helpers

## 🧩 Slurm helper functions

| Function | What it does |
|---|---|
| `validate_shell_script` | Checks the generated `.slurm` script for shell-syntax errors before spending a GPU allocation. |
| `submit_and_stream` | Runs `sbatch`, watches the job with `squeue`, streams the log, and finally reads accounting information from `sacct`. |

### Four Slurm commands you may see

- `sbatch` → submit a job
- `squeue` → ask whether a job is queued/running
- `sacct` → inspect the final job state/resources
- `srun` → launch the program/tasks inside an allocation

The long function is mostly **monitoring plumbing**. It does not change the neural network.


In [ ]:
# 🔒 RUN ONLY — syntax validation + submission monitoring

def validate_shell_script(path,):
    result = subprocess.run(["bash", "-n", str(path)], capture_output=True,
        text=True)

    if result.returncode != 0:
        print(result.stderr)
        return False

    print("✅ Shell syntax valid:", path.name)

    return True


def submit_and_stream(script_path, job_name, poll_seconds=2.0):
    # NERSC Jupyter may itself have a CUDA_VISIBLE_DEVICES
    # value. Do not pass that mask into a new batch job.
    submit_env = (os.environ.copy())

    inherited_gpu_env = {name: submit_env.get(name)
 for name in ["CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
            "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]
 if name in submit_env}

    for name in ["CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]:
        submit_env.pop(name, None)

    print("Notebook GPU environment:", inherited_gpu_env if inherited_gpu_env
        else "<none>")

    print("Submitting with inherited GPU " "visibility removed.")

    submit = subprocess.run(["sbatch", str(script_path)], capture_output=True,
        text=True, env=submit_env)

    if submit.returncode != 0:
        print(submit.stderr)

        raise RuntimeError("sbatch rejected the job.")

    submit_text = (submit.stdout.strip())

    job_id = (submit_text.split()[-1])

    output_file = (SLURM_LOG_DIR / f"{job_name}-{job_id}.out")

    print("Submitted job", job_id)

    print("Output:", output_file)

    last_size = 0

    while True:
        if output_file.exists():
            with output_file.open("r") as handle:
                handle.seek(last_size)

                text = handle.read()

                if text:
                    print(text, end="")

                last_size = (handle.tell())

        active = subprocess.run(["squeue", "-h", "-j", job_id],
            capture_output=True, text=True).stdout.strip()

        if not active:
            if output_file.exists():
                with output_file.open("r") as handle:
                    handle.seek(last_size)

                    text = (handle.read())

                    if text:
                        print(text, end="")

            break

        time.sleep(poll_seconds)

    summary = subprocess.run(["sacct", "-j", job_id, "--format="
            "JobID,State,ExitCode," "Elapsed,AllocTRES", "-n", "-P"],
        capture_output=True, text=True).stdout.strip()

    print("\n--- sacct summary ---")

    print(summary)

    main_state = None

    for line in (summary.splitlines()):
        fields = (line.split("|"))

        if (len(fields) >= 2 and fields[0] == job_id):
            main_state = (fields[1])
            break

    if (main_state is None or not main_state.startswith("COMPLETED")):
        raise RuntimeError(f"SLURM job {job_id} " f"finished with state "
            f"{main_state}.")

    print(f"✅ Job {job_id} " "completed successfully.")

    return job_id

# 9. Build the Maximum-Power Job

## Python environment preflight

Before generating the Slurm job, Notebook 3A verifies the exact Python interpreter that will run on the compute node.

The selected interpreter must successfully import:

```text
torch
transformers
scikit-learn
pandas
numpy
```

This prevents a common HPC problem:

```text
Jupyter kernel Python
        ≠
training-job Python
```

The printed **Slurm training Python** path is the one that will appear in the generated `.slurm` file.

In [ ]:
# 🔒 RUN ONLY — verify the selected Slurm Python one more time

preflight = subprocess.run([NOTEBOOK_PYTHON, "-c", ("import sys; "
            "import torch; " "import transformers; "
            "print('python:', sys.executable); "
            "print('torch:', torch.__version__); "
            "print('transformers:', transformers.__version__); "
            "print('CUDA build:', torch.version.cuda)")], capture_output=True,
    text=True)

print(preflight.stdout)

if preflight.returncode != 0:
    print(preflight.stderr)

    raise RuntimeError("Selected Slurm Python failed the PyTorch preflight.")

print("✅ Python environment preflight passed")

## 🧩 Why are we building command-line arguments?

The notebook and the compute-node training program are two separate Python processes.

```text
Jupyter notebook
   ↓ writes settings as command-line arguments
Slurm job
   ↓
maximum_dnabert.py reads those arguments
```

`build_max_arguments(...)` converts the `MAX_CONFIG` dictionary into the `--name value` format understood by `argparse` inside the training script.


In [ ]:
# 🔒 RUN ONLY — build command-line arguments

def build_max_arguments(config,):
    values = ["--data_dir", str(DATA_DIR), "--output_dir", str(RESULTS_DIR),
 "--run_name", str(config["run_name"]), "--epochs", str(config["epochs"]),
 "--batch_size", str(config["batch_size"]),
 "--learning_rate", str(config["learning_rate"]),
 "--weight_decay", str(config["weight_decay"]),
 "--warmup_ratio", str(config["warmup_ratio"]),
 "--dropout", str(config["dropout"]),
 "--max_length", str(config["max_length"]),
 "--precision", str(config["precision"]), "--seed", str(config["seed"])]

    return " ".join(shlex.quote(value) for value in values)

In [ ]:
# 🔒 RUN ONLY — generate one DDP Slurm job

def write_max_slurm_job(config, walltime="00:45:00", run_name=None):
    """Turn a config dict into a submittable .slurm file."""
    config = dict(config)
    if run_name is not None:
        config["run_name"] = run_name

    gpu_count = int(config["gpus"])
    plan = resolve_slurm(gpu_count)

    if config["batch_size"] % gpu_count != 0:
        raise ValueError(
            f"Global batch {config['batch_size']} is not divisible by "
            f"{gpu_count} GPUs — every GPU must get the same number of "
            f"examples."
        )

    arguments = build_max_arguments(config)
    job_name = ("nb3a-" + config["run_name"].replace("_", "-"))[:60]
    path = SCRIPTS_DIR / (config["run_name"] + ".slurm")

    # The #SBATCH header is built as a list so the reservation line is
    # plainly present or absent, instead of buried inside an f-string.
    header = [
        "#!/bin/bash",
        f"#SBATCH -A {NERSC_ACCOUNT}",
        "#SBATCH -C gpu",
        f"#SBATCH -q {plan['qos']}",
        f"#SBATCH -t {walltime}",
        "",
        f"#SBATCH -N {plan['nodes']}",
        f"#SBATCH --ntasks-per-node={plan['tasks_per_node']}",
        "#SBATCH --cpus-per-task=32",
        f"#SBATCH --gpus-per-node={plan['gpus_per_node']}",
        "#SBATCH --gpu-bind=none",
        "",
        f"#SBATCH -J {job_name}",
        f"#SBATCH -o {SLURM_LOG_DIR}/{job_name}-%j.out",
    ]
    if plan["reservation"]:
        header.insert(4, f"#SBATCH --reservation={plan['reservation']}")

    body = f"""
export SLURM_CPU_BIND="cores"

export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))

export NCCL_DEBUG=VERSION

echo "[PYTHON] training interpreter: {NOTEBOOK_PYTHON}"

{NOTEBOOK_PYTHON} -c "import sys, torch, transformers; print('[PYTHON]', sys.executable); print('[PYTORCH]', torch.__version__); print('[TRANSFORMERS]', transformers.__version__)"

if [ $? -ne 0 ]; then
    echo "[ERROR] Selected Python cannot import the DNABERT dependencies."
    exit 1
fi

echo "[BATCH] CUDA_VISIBLE_DEVICES before cleanup=${{CUDA_VISIBLE_DEVICES-<unset>}}"
echo "[BATCH] SLURM_JOB_GPUS=${{SLURM_JOB_GPUS-<unset>}}"

# Prevent the Jupyter server's GPU mask from restricting this job.
unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS

    echo "[SLURM→DDP] RANK=$RANK LOCAL_RANK=$LOCAL_RANK WORLD_SIZE=$WORLD_SIZE CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES-<unset>}}"

    {NOTEBOOK_PYTHON} {MAX_SCRIPT} {arguments}
'
"""

    path.write_text("\n".join(header) + "\n" + body)

    if not validate_shell_script(path):
        raise RuntimeError("Fix shell syntax before submitting.")

    return path, job_name

In [ ]:
# ✏️ RUN THIS — generate and inspect the actual job

MAX_SLURM_SCRIPT, MAX_JOB_NAME = (write_max_slurm_job(MAX_CONFIG,
        walltime="00:45:00"))

print(MAX_SLURM_SCRIPT.read_text())

### ✅ CHECKPOINT — before submitting

For a 2-GPU run, confirm:

```bash
#SBATCH --ntasks-per-node=2
#SBATCH --gpus-per-node=2
#SBATCH --gpu-bind=none
```

and:

```bash
RANK=$SLURM_PROCID
LOCAL_RANK=$SLURM_LOCALID
WORLD_SIZE=$SLURM_NTASKS
```

The training output should later show:

```text
rank 0 → cuda:0
rank 1 → cuda:1
```

# 10. Strong Scaling — Does 2× the GPUs Mean 2× the Speed?

In Notebook 1 you fine-tuned DNABERT on **one** GPU. Before we spend a big
reservation on a maximum run, let's measure what more GPUs actually buy.

### The measurement we are making

We keep the **global batch size fixed** and split it across more GPUs. So
with a global batch of 64:

```text
1 GPU  -> 64 examples per step on that GPU
2 GPUs -> 32 examples per step on each
4 GPUs -> 16 examples per step on each
```

The total arithmetic is identical. Only the number of workers changes. That
is **strong scaling**: same problem, more hardware, how much faster?

The other kind, **weak scaling**, keeps the per-GPU batch fixed so the
problem grows with the hardware. That is what the maximum run later does —
and it is why the maximum run is not a fair speed comparison.

### What to expect

Perfect scaling would be a straight line: 4 GPUs = 4× faster. You will not
get it. After every step, DDP must **all-reduce** the gradients so that all
GPUs agree on the update, and that communication is not free. Speedup falls
below the ideal line, and the gap grows with GPU count.

**Parallel efficiency** puts a number on it:

```text
efficiency = speedup / number_of_GPUs
```

1.0 is perfect. 0.8 means you are getting 80% of what you paid for.

In [ ]:
# ✏️ EDIT ME — which GPU counts to compare

# Short runs: we are timing the hardware, not training a good model.
SCALING_EPOCHS = 2

# The SAME total batch is split across however many GPUs — so this number
# must be divisible by every GPU count you test.
SCALING_GLOBAL_BATCH = 64

if RUN_MODE == "shared":
    SCALING_GPUS = [1, 2]
else:
    SCALING_GPUS = [1, 2, 4, 8]

for n in SCALING_GPUS:
    resolve_slurm(n)                       # fails early if a count is illegal
    if SCALING_GLOBAL_BATCH % n != 0:
        raise ValueError(f"{SCALING_GLOBAL_BATCH} is not divisible by {n}")

print("Will time these GPU counts:", SCALING_GPUS)
print(f"Fixed global batch: {SCALING_GLOBAL_BATCH}")
print()
for n in SCALING_GPUS:
    plan = resolve_slurm(n)
    print(f"  {n:>2} GPU(s): {plan['nodes']} node(s), "
          f"{SCALING_GLOBAL_BATCH // n} examples per GPU per step")

In [ ]:
# ✏️ RUN THIS — submit one short job per GPU count (runs one after another)

scaling_runs = []

for n_gpus in SCALING_GPUS:
    run_name = f"scaling_{n_gpus}gpu"
    config = dict(MAX_CONFIG)
    config.update({"gpus": n_gpus, "epochs": SCALING_EPOCHS,
                   "batch_size": SCALING_GLOBAL_BATCH, "run_name": run_name})

    print("=" * 60)
    print(f"  {n_gpus} GPU(s)  ->  {run_name}")
    print("=" * 60)

    script_path, job_name = write_max_slurm_job(config, walltime="00:20:00")
    submit_and_stream(script_path, job_name)
    scaling_runs.append((n_gpus, run_name))

print()
print("✅ All scaling jobs finished.")

In [ ]:
# 🔒 RUN ONLY — collect the timings into one table

rows = []
for n_gpus, run_name in scaling_runs:
    summary_path = RESULTS_DIR / (run_name + "_summary.json")
    if not summary_path.exists():
        print(f"⚠️  missing: {summary_path.name} — did that job fail?")
        continue
    with open(summary_path) as handle:
        summary = json.load(handle)
    rows.append({
        "gpus": n_gpus,
        "seconds": summary["training_time_seconds"],
        "examples_per_second": summary["examples_per_second"],
        "peak_gpu_memory_gb": summary["peak_gpu_memory_gb"],
        "best_val_auroc": summary["best_val_auroc"],
    })

scaling = pd.DataFrame(rows).sort_values("gpus").reset_index(drop=True)

baseline = scaling.loc[scaling["gpus"] == scaling["gpus"].min(), "seconds"]
baseline = float(baseline.iloc[0])

scaling["speedup"] = baseline / scaling["seconds"]
scaling["ideal_speedup"] = scaling["gpus"] / scaling["gpus"].min()
scaling["efficiency"] = scaling["speedup"] / scaling["ideal_speedup"]

scaling.round(3)

In [ ]:
# 👀 READ — three views of the same measurement

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(scaling["gpus"], scaling["seconds"], marker="o")
axes[0].set_xlabel("GPUs")
axes[0].set_ylabel("Training time (seconds)")
axes[0].set_title("Wall-clock time")
axes[0].set_ylim(bottom=0)

axes[1].plot(scaling["gpus"], scaling["ideal_speedup"], linestyle="--",
             label="Ideal (linear)")
axes[1].plot(scaling["gpus"], scaling["speedup"], marker="o", label="Measured")
axes[1].set_xlabel("GPUs")
axes[1].set_ylabel("Speedup vs fewest GPUs")
axes[1].set_title("Speedup — the gap is communication")
axes[1].legend()

axes[2].bar(scaling["gpus"].astype(str), scaling["efficiency"])
axes[2].axhline(1.0, linestyle="--", label="Perfect")
axes[2].set_xlabel("GPUs")
axes[2].set_ylabel("Parallel efficiency")
axes[2].set_title("Fraction of what you paid for")
axes[2].set_ylim(0, 1.15)
axes[2].legend()

plt.tight_layout()
plt.show()

### ✅ CHECKPOINT — read your own scaling numbers

1. How many times faster was your largest GPU count than your smallest?
2. Was that equal to the GPU ratio? If not, where did the missing time go?
3. Look at the `efficiency` column. At which GPU count does it start to
   drop noticeably?
4. Look at `best_val_auroc` across the runs. The models trained on 1 GPU and
   on 8 GPUs did the *same* arithmetic — should their AUROC be identical?
   Why might it differ slightly anyway?

**A useful habit:** more GPUs is not automatically better. There is a point
where adding hardware buys you almost nothing, and on a shared machine that
wasted allocation is time someone else could have used. Your efficiency plot
tells you where that point is *for this model and this dataset*.

### 🧪 Try it yourself

This dataset is small — roughly 1,800 sequences. Communication cost per step
stays about the same no matter how big the batch is, so a bigger batch
spreads that cost over more work. Raise `SCALING_GLOBAL_BATCH` to 256 and
re-run. Does efficiency at your largest GPU count improve?

# 11. Launch the Maximum DNABERT Run

### Environment checkpoint

Before DDP starts, the Slurm output should print something like:

```text
[PYTHON] /.../dna-llm/bin/python
[PYTORCH] <version>
[TRANSFORMERS] <version>
```

If you instead see:

```text
ModuleNotFoundError: No module named 'torch'
```

the job is using the wrong Python interpreter and should be stopped before debugging DDP or the model.

In [ ]:
# ✏️ RUN THIS

MAX_JOB_ID = submit_and_stream(MAX_SLURM_SCRIPT, MAX_JOB_NAME)

### What success should look like

Early in the log:

```text
=== MAXIMUM DNABERT ===
GPUs: 2
Precision: bf16
Global batch: 64
Local batch/GPU: 32
Trainable percent: 100.00%
```

Then each epoch prints training and validation metrics.

At the end:

```text
=== FINAL REPORT ===
Best AUROC: ...
Training time: ...
Examples/sec: ...
Peak GPU memory: ... / ... GB (...%)
Checkpoint: ...
```

# 12. Load the Saved Maximum-Fine-Tuning Results

In [ ]:
# 🔒 RUN ONLY

RUN_NAME = (MAX_CONFIG["run_name"])

SUMMARY_PATH = (RESULTS_DIR / (RUN_NAME + "_summary.json"))

HISTORY_PATH = (RESULTS_DIR / (RUN_NAME + "_history.csv"))

PARAMETER_GROUPS_PATH = (RESULTS_DIR / (RUN_NAME + "_parameter_groups.csv"))

if not SUMMARY_PATH.exists():
    raise FileNotFoundError("The maximum DNABERT run "
        "has not produced a summary yet.")

with open(SUMMARY_PATH) as handle:
    max_summary = (json.load(handle))

max_history = (pd.read_csv(HISTORY_PATH))

parameter_groups = (pd.read_csv(PARAMETER_GROUPS_PATH))

pd.Series(max_summary, name="value")

# 13. Parameter Audit

In [ ]:
# 👀 READ — useful model parameters should be fully trainable

parameter_report = pd.DataFrame({"metric": ["Trainable parameters",
        "Total parameters", "Trainable percent"],
 "value": [max_summary["trainable_parameters"],
 max_summary["total_parameters"], max_summary["trainable_percent"]]})

parameter_report

In [ ]:
# 👀 READ — trainable parameter count by model component

parameter_plot = (parameter_groups[parameter_groups["trainable"] == True]
    .sort_values("numel"))

plt.figure(figsize=(9, 6))

plt.barh(parameter_plot["group"], parameter_plot["numel"])

plt.xlabel("Trainable parameters")

plt.title("Where DNABERT's Trainable Parameters Live")

plt.tight_layout()
plt.show()

### ✅ CHECKPOINT — what should you notice?

Most trainable parameters should be inside the **Transformer encoder layers**, not the small classification head.

That is what makes this a true **full fine-tuning** experiment.

# 14. Training Curves

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(max_history["epoch"], max_history["train_loss"], marker="o",
    label="Train loss")

plt.plot(max_history["epoch"], max_history["val_loss"], marker="o",
    label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Maximum DNABERT — Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(max_history["epoch"], max_history["val_auroc"], marker="o")

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.title("Maximum DNABERT — AUROC")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(max_history["epoch"], max_history["epoch_seconds"], marker="o")

plt.xlabel("Epoch")
plt.ylabel("Seconds")
plt.title("Time per Epoch")
plt.tight_layout()
plt.show()

# 15. Did We Actually Use the GPU Heavily?

In [ ]:
gpu_report = pd.DataFrame({"metric": ["GPUs", "Precision", "Global batch",
        "Local batch / GPU", "Training time (s)", "Examples / second",
        "Peak GPU memory (GB)", "GPU capacity (GB)", "Peak memory (%)"],
 "value": [max_summary["num_gpus"], max_summary["precision"],
 max_summary["global_batch_size"], max_summary["local_batch_size"],
 max_summary["training_time_seconds"], max_summary["examples_per_second"],
 max_summary["peak_gpu_memory_gb"], max_summary["gpu_memory_capacity_gb"],
 max_summary["peak_gpu_memory_percent"]]})

gpu_report

In [ ]:
# 👀 READ — simple GPU-memory utilization visualization

plt.figure(figsize=(6, 4))

plt.bar(["Peak allocated memory"], [max_summary["peak_gpu_memory_percent"]])

plt.axhline(100, linestyle="--")

plt.ylabel("% of GPU memory capacity")

plt.ylim(0, 105)

plt.title("Maximum DNABERT GPU Memory Use")

plt.tight_layout()
plt.show()

## How to interpret GPU memory

Peak memory is **not the same thing as GPU compute utilization**.

But it gives us a useful tuning signal.

### If peak memory is low

For example:

```text
< 50%
```

you may be able to increase:

```python
LOCAL_BATCH_SIZE
```

and give each GPU more examples per step.

### If peak memory is very high

For example:

```text
> 90%
```

you are close to the memory limit.

Increasing the batch further could cause an out-of-memory error.

### The goal is not “100% memory at all costs”

The goal is:

> **high useful throughput without crashing or degrading the experiment.**

# 16. Final Model Report

In [ ]:
final_report = pd.DataFrame({"metric": ["Best validation AUROC", "Best epoch",
        "Final accuracy", "Final F1", "Final AUROC", "Final AUPRC",
        "Trainable %", "Training time (s)", "Examples / second"],
 "value": [max_summary["best_val_auroc"], max_summary["best_epoch"],
 max_summary["final_val_accuracy"], max_summary["final_val_f1"],
 max_summary["final_val_auroc"], max_summary["final_val_auprc"],
 max_summary["trainable_percent"], max_summary["training_time_seconds"],
 max_summary["examples_per_second"]]})

final_report

In [ ]:
print("Saved trained model:")

print(max_summary["checkpoint_path"])

# 17. Can You Push the GPU Harder?

Do **not** change many things at once.

The simplest hardware-utilization experiment is the local batch size.

Current:

```python
LOCAL_BATCH_SIZE = 32
```

If the previous run has substantial memory headroom, try:

```python
LOCAL_BATCH_SIZE = 48
```

or:

```python
LOCAL_BATCH_SIZE = 64
```

Then recompute:

```python
GLOBAL_BATCH_SIZE = (
    GPU_COUNT
    * LOCAL_BATCH_SIZE
)
```

and create a **new run name**.

Example:

```python
"run_name": "dnabert_maximum_batch64"
```

### What should you record?

```text
local batch
global batch
peak GPU memory
examples / second
training time
AUROC
```

Increasing batch size is useful only if the extra throughput does not destroy the scientific result.

### 🧠 Student interpretation questions

1. What percentage of the useful model parameters were trainable?
2. Which part of DNABERT contains most of the trainable parameters?
3. How many examples did each GPU process per step?
4. Why do we use BF16 instead of simply converting every model parameter to BF16?
5. What was the peak GPU memory percentage?
6. Could you safely increase the local batch size?
7. Did validation AUROC continue improving for all epochs?
8. Which epoch had the best AUROC?
9. Why might the final epoch not be the best epoch?
10. What is the difference between:
   - maximizing **trainable parameters**, and
   - maximizing **GPU utilization**?

# 18. Notebook 3A Summary

Notebook 3A intentionally does **one thing**:

> **Use the available GPUs to fully fine-tune one DNABERT model.**

The recipe is:

```text
DNA sequences
↓
overlapping 6-mers
↓
pretrained DNABERT-6
↓
remove unused BERT pooler
↓
100% of remaining parameters trainable
↓
BF16 autocast
↓
large local batch on every GPU
↓
DDP gradient synchronization
↓
full fine-tuning
↓
save final trained checkpoint
```

The main hardware measurements are:

```text
training time
examples / second
peak GPU memory
GPU count
```

The main scientific measurements are:

```text
AUROC
AUPRC
accuracy
F1
```

A successful HPC workflow needs **both**:

> fast computation **and** a scientifically useful model.

# ✅ End of Notebook 3A